# 02 - Forward Propagation and Activation Functions

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Express forward propagation in **matrix form**: $\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$, $\mathbf{a} = \sigma(\mathbf{z})$
- Implement and visualize the major **activation functions**: Sigmoid, Tanh, ReLU, Leaky ReLU, GELU
- Build a complete **forward pass** through a multi-layer network using only NumPy
- Apply practical guidelines for **choosing activation functions**

## Prerequisites

- Completed [01 - What is a Neural Network?](01_What_is_a_Neural_Network.ipynb)
- NumPy array operations, matrix multiplication
- Basic calculus (derivatives) -- helpful but not required for this notebook

## Table of Contents

1. [Forward Propagation in Matrix Form](#1-forward-propagation-in-matrix-form)
2. [Activation Functions: Theory and Code](#2-activation-functions-theory-and-code)
   - 2.1 [Sigmoid](#21-sigmoid)
   - 2.2 [Tanh](#22-tanh)
   - 2.3 [ReLU](#23-relu)
   - 2.4 [Leaky ReLU](#24-leaky-relu)
   - 2.5 [GELU](#25-gelu)
3. [Plotting All Activations](#3-plotting-all-activations)
4. [Full Forward Pass: 2-Layer Network](#4-full-forward-pass-2-layer-network)
5. [When to Use Which Activation](#5-when-to-use-which-activation)
6. [Exercise: 3-Layer Forward Pass](#6-exercise-3-layer-forward-pass)
7. [Common Mistakes & Debugging Tips](#7-common-mistakes--debugging-tips)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Forward Propagation in Matrix Form

**Forward propagation** is the process of passing input data through the network, layer by layer, to produce an output.

For a single layer with $m$ neurons receiving input $\mathbf{x} \in \mathbb{R}^n$:

$$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$$
$$\mathbf{a} = \sigma(\mathbf{z})$$

where:
- $\mathbf{W} \in \mathbb{R}^{m \times n}$ is the weight matrix
- $\mathbf{b} \in \mathbb{R}^{m}$ is the bias vector
- $\sigma$ is the activation function (applied element-wise)
- $\mathbf{z}$ is the **pre-activation** (linear combination)
- $\mathbf{a}$ is the **post-activation** (layer output)

For a multi-layer network, we chain these operations:

$$\mathbf{z}^{[1]} = \mathbf{W}^{[1]}\mathbf{x} + \mathbf{b}^{[1]}, \quad \mathbf{a}^{[1]} = \sigma^{[1]}(\mathbf{z}^{[1]})$$
$$\mathbf{z}^{[2]} = \mathbf{W}^{[2]}\mathbf{a}^{[1]} + \mathbf{b}^{[2]}, \quad \mathbf{a}^{[2]} = \sigma^{[2]}(\mathbf{z}^{[2]})$$
$$\vdots$$
$$\hat{\mathbf{y}} = \mathbf{a}^{[L]}$$

The superscript $[l]$ denotes the layer index. Each layer can use a different activation function $\sigma^{[l]}$.

### Batched Forward Propagation

In practice, we process a **batch** of $B$ samples simultaneously. The input becomes a matrix $\mathbf{X} \in \mathbb{R}^{B \times n}$ (each row is one sample):

$$\mathbf{Z} = \mathbf{X} \mathbf{W}^T + \mathbf{b}$$

Note the transpose on $\mathbf{W}$ -- this is because we store samples as rows. NumPy broadcasting handles adding $\mathbf{b}$ to each row.

---
## 2. Activation Functions: Theory and Code

Activation functions introduce **nonlinearity** into the network. Without them, any composition of linear layers is still linear, and the network could only learn linear mappings.

### 2.1 Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Properties:**
- Output range: $(0, 1)$
- Smooth, differentiable everywhere
- Derivative: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$
- **Drawback:** Saturates for large $|z|$ (gradients vanish), outputs not zero-centered

In [ ]:
def sigmoid(z):
    """Sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_derivative(z):
    """Derivative of sigmoid."""
    s = sigmoid(z)
    return s * (1 - s)


# Quick demo
z_test = np.array([-5, -1, 0, 1, 5])
print("z:         ", z_test)
print("sigmoid(z):", np.round(sigmoid(z_test), 4))

### 2.2 Tanh

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

**Properties:**
- Output range: $(-1, 1)$ -- zero-centered (often better than sigmoid for hidden layers)
- Derivative: $\tanh'(z) = 1 - \tanh^2(z)$
- **Drawback:** Still saturates for large $|z|$ (vanishing gradients)

In [ ]:
def tanh_activation(z):
    """Tanh activation function."""
    return np.tanh(z)


def tanh_derivative(z):
    """Derivative of tanh."""
    return 1.0 - np.tanh(z) ** 2


print("z:      ", z_test)
print("tanh(z):", np.round(tanh_activation(z_test), 4))

### 2.3 ReLU

$$\text{ReLU}(z) = \max(0, z)$$

**Properties:**
- Output range: $[0, \infty)$
- Computationally cheap (just a threshold)
- Does not saturate for positive values -- helps with vanishing gradient problem
- Derivative: $1$ if $z > 0$, else $0$
- **Drawback:** "Dying ReLU" problem -- neurons with negative pre-activation output 0 and receive 0 gradient, potentially never recovering

In [ ]:
def relu(z):
    """ReLU activation function."""
    return np.maximum(0, z)


def relu_derivative(z):
    """Derivative of ReLU (subgradient at 0 defined as 0)."""
    return (z > 0).astype(float)


print("z:      ", z_test)
print("relu(z):", relu(z_test))

### 2.4 Leaky ReLU

$$\text{LeakyReLU}(z) = \begin{cases} z & \text{if } z > 0 \\ \alpha z & \text{if } z \leq 0 \end{cases}$$

where $\alpha$ is a small positive constant (commonly $0.01$).

**Properties:**
- Addresses the dying ReLU problem by allowing a small gradient for negative inputs
- Output range: $(-\infty, \infty)$
- Derivative: $1$ if $z > 0$, else $\alpha$

In [ ]:
def leaky_relu(z, alpha=0.01):
    """Leaky ReLU activation function."""
    return np.where(z > 0, z, alpha * z)


def leaky_relu_derivative(z, alpha=0.01):
    """Derivative of Leaky ReLU."""
    return np.where(z > 0, 1.0, alpha)


print("z:             ", z_test)
print("leaky_relu(z): ", leaky_relu(z_test))

### 2.5 GELU

$$\text{GELU}(z) = z \cdot \Phi(z) \approx 0.5 \, z \left(1 + \tanh\left[\sqrt{\frac{2}{\pi}}\left(z + 0.044715 \, z^3\right)\right]\right)$$

where $\Phi(z)$ is the cumulative distribution function of the standard normal distribution.

**Properties:**
- Used extensively in modern architectures (BERT, GPT, Vision Transformers)
- Smooth approximation of ReLU
- Non-monotonic for negative values (allows small negative outputs)
- More expensive to compute than ReLU

In [ ]:
def gelu(z):
    """GELU activation function (tanh approximation)."""
    return 0.5 * z * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (z + 0.044715 * z**3)))


print("z:      ", z_test)
print("gelu(z):", np.round(gelu(z_test), 4))

---
## 3. Plotting All Activations

Let us visualize all activation functions and their derivatives side by side.

In [ ]:
z = np.linspace(-6, 6, 500)

activations = {
    'Sigmoid': (sigmoid, sigmoid_derivative),
    'Tanh': (tanh_activation, tanh_derivative),
    'ReLU': (relu, relu_derivative),
    'Leaky ReLU': (leaky_relu, leaky_relu_derivative),
    'GELU': (gelu, None),  # derivative omitted for brevity
}

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot activation functions ---
ax = axes[0]
for (name, (fn, _)), color in zip(activations.items(), colors):
    ax.plot(z, fn(z), label=name, color=color, linewidth=2)

ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('z', fontsize=12)
ax.set_ylabel('activation(z)', fontsize=12)
ax.set_title('Activation Functions', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-2, 6)
ax.grid(True, alpha=0.3)

# --- Plot derivatives ---
ax = axes[1]
derivs = {
    'Sigmoid': (sigmoid_derivative, '#e74c3c'),
    'Tanh': (tanh_derivative, '#3498db'),
    'ReLU': (relu_derivative, '#2ecc71'),
    'Leaky ReLU': (leaky_relu_derivative, '#9b59b6'),
}
for name, (fn, color) in derivs.items():
    ax.plot(z, fn(z), label=name, color=color, linewidth=2)

ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('z', fontsize=12)
ax.set_ylabel("activation'(z)", fontsize=12)
ax.set_title('Activation Derivatives', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-0.2, 1.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observations from the plots:**

- **Sigmoid and Tanh** saturate at both extremes -- their derivatives approach 0 for large $|z|$, causing the vanishing gradient problem.
- **ReLU** has a constant derivative of 1 for positive inputs (no saturation), but exactly 0 for negative inputs.
- **Leaky ReLU** fixes the dead-neuron issue by allowing a small non-zero slope for negative inputs.
- **GELU** is smooth everywhere and closely resembles ReLU for large positive values.

---
## 4. Full Forward Pass: 2-Layer Network

Let us implement a complete forward pass through a 2-layer network (1 hidden layer + 1 output layer) in NumPy.

**Architecture:** 3 inputs $\rightarrow$ 4 hidden (ReLU) $\rightarrow$ 1 output (sigmoid)

This is suitable for a binary classification task.

In [ ]:
def forward_pass_2layer(X, W1, b1, W2, b2):
    """
    Forward pass through a 2-layer network.
    
    Architecture: Input -> Hidden (ReLU) -> Output (Sigmoid)
    
    Parameters
    ----------
    X  : np.ndarray, shape (batch_size, n_input)
    W1 : np.ndarray, shape (n_hidden, n_input)
    b1 : np.ndarray, shape (n_hidden,)
    W2 : np.ndarray, shape (n_output, n_hidden)
    b2 : np.ndarray, shape (n_output,)
    
    Returns
    -------
    cache : dict with intermediate values (useful for backprop later)
    """
    # Layer 1: Hidden layer with ReLU
    Z1 = X @ W1.T + b1          # (batch, n_hidden)
    A1 = relu(Z1)               # (batch, n_hidden)
    
    # Layer 2: Output layer with Sigmoid
    Z2 = A1 @ W2.T + b2         # (batch, n_output)
    A2 = sigmoid(Z2)            # (batch, n_output)
    
    cache = {
        'X': X, 'Z1': Z1, 'A1': A1,
        'Z2': Z2, 'A2': A2,
    }
    return A2, cache


# Initialize network
np.random.seed(42)
n_input, n_hidden, n_output = 3, 4, 1

# He initialization for ReLU layer, Xavier for sigmoid layer
W1 = np.random.randn(n_hidden, n_input) * np.sqrt(2.0 / n_input)
b1 = np.zeros(n_hidden)
W2 = np.random.randn(n_output, n_hidden) * np.sqrt(1.0 / n_hidden)
b2 = np.zeros(n_output)

# Create a small batch of data
X = np.array([
    [0.5, -0.3, 0.8],
    [1.0,  0.2, -0.5],
    [-0.7, 0.9, 0.1],
])

# Forward pass
output, cache = forward_pass_2layer(X, W1, b1, W2, b2)

print("Input X (3 samples, 3 features):")
print(X)
print(f"\nW1 shape: {W1.shape}, b1 shape: {b1.shape}")
print(f"W2 shape: {W2.shape}, b2 shape: {b2.shape}")
print(f"\nHidden pre-activation Z1 shape: {cache['Z1'].shape}")
print(f"Hidden activation A1 shape:     {cache['A1'].shape}")
print(f"Output Z2 shape:                {cache['Z2'].shape}")
print(f"\nPredictions (probabilities):")
for i in range(len(X)):
    print(f"  Sample {i}: {output[i, 0]:.4f}")

In [ ]:
# Step-by-step trace for sample 0
print("=== Tracing forward pass for sample 0 ===")
x0 = X[0]  # shape (3,)
print(f"\n1. Input:  x = {x0}")

z1_0 = W1 @ x0 + b1
print(f"\n2. Hidden pre-activation:  z1 = W1 @ x + b1 = {z1_0}")

a1_0 = relu(z1_0)
print(f"3. Hidden activation:      a1 = ReLU(z1)   = {a1_0}")

z2_0 = W2 @ a1_0 + b2
print(f"\n4. Output pre-activation:  z2 = W2 @ a1 + b2 = {z2_0}")

a2_0 = sigmoid(z2_0)
print(f"5. Output activation:      a2 = sigmoid(z2)  = {a2_0}")
print(f"\nMatches batch result: {np.allclose(a2_0, output[0])}")

---
## 5. When to Use Which Activation

Here is a practical decision guide:

| Activation | Use Case | Avoid When |
|---|---|---|
| **ReLU** | Default for hidden layers. Fast, effective, works well in most cases. | N/A (safe default) |
| **Leaky ReLU** | When you observe many "dead" neurons (outputs stuck at 0) | N/A (good ReLU alternative) |
| **GELU** | Transformer-based architectures (BERT, GPT, ViT) | When computational cost matters on limited hardware |
| **Sigmoid** | **Output layer** for binary classification (probability in (0,1)) | Hidden layers (vanishing gradients) |
| **Tanh** | Hidden layers in RNNs (zero-centered output helps). Output when you need range (-1,1). | Deep feedforward networks (vanishing gradients) |
| **Softmax** | **Output layer** for multi-class classification | Hidden layers |
| **Linear (none)** | **Output layer** for regression | Hidden layers (makes network trivially linear) |

**Rule of thumb:**
- Hidden layers: **ReLU** (or Leaky ReLU / GELU)
- Output layer: depends on the task (sigmoid, softmax, or linear)

---
## 6. Exercise: 3-Layer Forward Pass

**Task:** Implement a forward pass for a 3-layer network (2 hidden layers + 1 output layer).

**Architecture:** 4 inputs $\rightarrow$ 8 hidden (ReLU) $\rightarrow$ 4 hidden (ReLU) $\rightarrow$ 1 output (sigmoid)

```python
def forward_pass_3layer(X, W1, b1, W2, b2, W3, b3):
    """
    Forward pass through a 3-layer network.
    
    Architecture:
        Input(4) -> Hidden1(8, ReLU) -> Hidden2(4, ReLU) -> Output(1, Sigmoid)
    
    Returns: output predictions, cache dict
    """
    # TODO: implement the forward pass
    # Layer 1: ...
    # Layer 2: ...
    # Layer 3: ...
    pass

# Test with random data
np.random.seed(42)
X_test = np.random.randn(5, 4)  # 5 samples, 4 features

# TODO: initialize weights and biases, run forward pass, print output
```

In [ ]:
# Try the exercise yourself before scrolling down!











### Solution

In [ ]:
# ----- Solution -----

def forward_pass_3layer(X, W1, b1, W2, b2, W3, b3):
    """
    Forward pass through a 3-layer network.
    
    Architecture:
        Input(4) -> Hidden1(8, ReLU) -> Hidden2(4, ReLU) -> Output(1, Sigmoid)
    """
    # Layer 1: Hidden layer 1 with ReLU
    Z1 = X @ W1.T + b1
    A1 = relu(Z1)
    
    # Layer 2: Hidden layer 2 with ReLU
    Z2 = A1 @ W2.T + b2
    A2 = relu(Z2)
    
    # Layer 3: Output layer with Sigmoid
    Z3 = A2 @ W3.T + b3
    A3 = sigmoid(Z3)
    
    cache = {
        'X': X,
        'Z1': Z1, 'A1': A1,
        'Z2': Z2, 'A2': A2,
        'Z3': Z3, 'A3': A3,
    }
    return A3, cache


# Initialize network with He initialization
np.random.seed(42)
n_in, n_h1, n_h2, n_out = 4, 8, 4, 1

W1 = np.random.randn(n_h1, n_in) * np.sqrt(2.0 / n_in)
b1 = np.zeros(n_h1)
W2 = np.random.randn(n_h2, n_h1) * np.sqrt(2.0 / n_h1)
b2 = np.zeros(n_h2)
W3 = np.random.randn(n_out, n_h2) * np.sqrt(1.0 / n_h2)
b3 = np.zeros(n_out)

# Test data
X_test = np.random.randn(5, 4)

# Forward pass
output, cache = forward_pass_3layer(X_test, W1, b1, W2, b2, W3, b3)

print("Network architecture: 4 -> 8 (ReLU) -> 4 (ReLU) -> 1 (Sigmoid)")
print(f"\nInput shape:   {X_test.shape}")
print(f"Output shape:  {output.shape}")
print(f"\nLayer shapes:")
print(f"  Z1: {cache['Z1'].shape}, A1: {cache['A1'].shape}")
print(f"  Z2: {cache['Z2'].shape}, A2: {cache['A2'].shape}")
print(f"  Z3: {cache['Z3'].shape}, A3: {cache['A3'].shape}")
print(f"\nPredictions:")
for i in range(len(X_test)):
    print(f"  Sample {i}: p = {output[i, 0]:.4f}")

---
## 7. Common Mistakes & Debugging Tips

**1. Using sigmoid/tanh in hidden layers of deep networks**
- Leads to vanishing gradients. Gradients shrink exponentially as they propagate back through layers.
- Fix: Use ReLU (or variants) in hidden layers.

**2. Shape errors in batched forward pass**
- Remember: `X @ W.T + b` when X is `(batch, n_in)` and W is `(n_out, n_in)`.
- Tip: Print shapes at every step when debugging.

**3. Forgetting to apply activation**
- Easy to compute `Z = X @ W.T + b` and accidentally pass `Z` to the next layer instead of `A = activation(Z)`.
- This makes the network purely linear.

**4. Incorrect weight initialization**
- Using `np.random.randn(...)` without scaling leads to exploding or vanishing activations.
- Use He initialization (`* sqrt(2/n_in)`) for ReLU layers.
- Use Xavier/Glorot initialization (`* sqrt(1/n_in)` or `* sqrt(2/(n_in+n_out))`) for sigmoid/tanh.

**5. Numerical overflow in sigmoid**
- `np.exp(-z)` overflows for very large negative `z`.
- Numerically stable version:
  ```python
  def sigmoid_stable(z):
      return np.where(z >= 0,
                      1 / (1 + np.exp(-z)),
                      np.exp(z) / (1 + np.exp(z)))
  ```

**6. Confusing activation for output layer vs hidden layers**
- Hidden: ReLU/LeakyReLU
- Output: depends on task (sigmoid for binary, softmax for multi-class, linear for regression)

---

**Next notebook:** [03 - Backpropagation and Autograd Intuition](03_Backpropagation_and_Autograd_Intuition.ipynb) -- we learn how the network adjusts its weights using the chain rule and gradient descent.